# Lab 27: AI Red Teaming as a Regression Loop

*AI Cybersecurity Masterclass · Lab 27 · AI60secLab*

**Companion video:** [▶ How AI Red Teaming Tests AI Safety in 60 Seconds](https://www.youtube.com/watch?v=_KCq0iEtnzk&list=PLG4l6Ssi9A00)

**Research question:** How do you know an AI safety control works before a real failure proves it does not?

We red-team the full application from Labs 23 and 24: the assistant, the agent, and their policies. Every case states its expected safe behavior before it runs. Failures are prioritized, fixed, retested, and kept as regression tests.

**Learning objectives**
- write test cases with the expected safe behavior defined up front;
- test the whole application, not only the model;
- prioritize failures by consequence and reach;
- check whether monitoring noticed each test;
- turn every fixed failure into a permanent regression test.

> **About the simulated assistant.** These labs use `SimulatedModel`, a deterministic stand-in for an LLM. It reproduces a documented failure of real models: it understands instructions written in many phrasings, including ones hidden in retrieved data, and proposes actions based on them. That keeps every result repeatable and free to run. The *controls* you build and measure are real code that could sit in front of a real model unchanged. What the simulation cannot tell you is how often a specific production model falls for a specific payload: that needs testing against that model.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

from src.redteam import APP_VERSIONS, default_cases, run_campaign

cases = default_cases()
display(pd.DataFrame([
    {"case_id": c.case_id, "target": c.target, "category": c.category,
     "expected_safe_behavior": c.expected_safe_behavior,
     "consequence": c.consequence, "reach": c.reach}
    for c in cases
]))

## Step 1: Red-team v1

v1 has a prompt-pattern filter and an autonomous agent. Expected behavior was written before running anything.

In [ ]:
versions = list(APP_VERSIONS)
v1 = run_campaign(cases, versions[0])
print(versions[0], "| pass rate:", round(v1["passed"].mean(), 2))
display(v1[["case_id", "category", "passed", "monitoring_noticed", "priority"]])

## Step 2: Prioritize by consequence and reach

Fix order comes from impact (consequence × reach), not from how clever the test looked.

In [ ]:
backlog = v1[~v1["passed"]].sort_values("priority", ascending=False)
display(backlog[["case_id", "category", "priority", "evidence"]])

## Step 3: Fix and retest (v2)

v2 adds the layered controls from Lab 23 and the policy-gated agent from Lab 24.

In [ ]:
v2 = run_campaign(cases, versions[1])
print(versions[1], "| pass rate:", round(v2["passed"].mean(), 2))
display(v2[~v2["passed"]][["case_id", "category", "expected_safe_behavior", "evidence"]])

One case still fails: an attacker-supplied image link carries the summary to an outside domain when the answer is rendered. The secret filter never fires because no secret is involved, so **monitoring did not notice either**. Check:

In [ ]:
display(v2[v2["case_id"] == "RT-exfil-markdown-summary"][["case_id", "passed", "monitoring_noticed"]])

## Step 4: Fix again (v3) and run the full suite as regression

v3 strips links to non-allowlisted domains from answers and logs each removal. Every earlier failure stays in the suite forever.

In [ ]:
v3 = run_campaign(cases, versions[2])
history = pd.concat([v1, v2, v3])
history["release"] = history["version"].str.split(" ").str[0]
regression = history.pivot_table(index="case_id", columns="release", values="passed", aggfunc="max").astype(int)
display(regression[["v1", "v2", "v3"]])

summary = history.groupby("release", sort=False).agg(
    pass_rate=("passed", "mean"),
    noticed_rate=("monitoring_noticed", lambda s: s.dropna().astype(bool).mean()),
)
display(summary.round(2))

## Interpretation

1. Why must the expected safe behavior be written before the test runs?
2. The over-blocking case tests that safety did not break the legitimate job. Why does a red-team suite need cases like that?
3. Which v2 failure would a model-only test have missed, and why?
4. A test that passes but goes unnoticed by monitoring: is that a finding? What would you add?

**Takeaway:** a fixed failure becomes a regression test. Measure whether controls passed, whether monitoring noticed, and whether the fix stayed fixed.